In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
class DGMNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers):
        super(DGMNet, self).__init__()

        self.input = nn.Linear(input_dim, hidden_dim)
        self.hidden = nn.ModuleList(
            [nn.Linear(hidden_dim, hidden_dim) for _ in range(n_layers)]
        )
        self.output = nn.Linear(hidden_dim, 1)

    def activation(self, x):
        # Swish / SiLU activation (smooth and good for PDEs)
        return x * torch.sigmoid(x)

    def forward(self, t, x):
        # concatenate time and space
        inp = torch.cat([t, x], dim=1)

        h = self.activation(self.input(inp))
        for layer in self.hidden:
            h = self.activation(layer(h))

        return self.output(h)

class DeepGalerkinSolver:
    def __init__(self, H, M, C, D, R, sigma, alpha,
                 T,
                 hidden_dim,
                 n_layers,
                 device):

        self.device = device
        self.T = T

        # Convert matrices to tensors
        self.H = torch.tensor(H, dtype=torch.float32, device=self.device)
        self.M = torch.tensor(M, dtype=torch.float32, device=self.device)
        self.C = torch.tensor(C, dtype=torch.float32, device=self.device)
        self.D = torch.tensor(D, dtype=torch.float32, device=self.device)
        self.R = torch.tensor(R, dtype=torch.float32, device=self.device)
        self.sigma = torch.tensor(sigma, dtype=torch.float32, device=self.device)
        self.alpha = torch.tensor(alpha, dtype=torch.float32, device=self.device)

        self._validate_positive_definite(self.C, "C")
        self._validate_positive_definite(self.R, "R")
        self._validate_strictly_positive_definite(self.D, "D")

        self.dim = self.H.shape[0]

        self.net = DGMNet(self.dim + 1, hidden_dim, n_layers).to(self.device)

        self.optimizer = optim.Adam(self.net.parameters(), lr=1e-3)

    def _validate_positive_definite(self, A, name):
        """
        Checks if A is symmetric positive definite.
        Raises ValueError if not.
        """

        # Check symmetry
        if not torch.allclose(A, A.T, atol=1e-6):
            raise ValueError(f"Matrix {name} is not symmetric.")

        # Cholesky test
        try:
            torch.linalg.cholesky(A)
        except RuntimeError:
            raise ValueError(f"Matrix {name} is not positive definite.")

    def _validate_strictly_positive_definite(self, A, name):
        """
        Checks if A is strictly positive definite
        (all eigenvalues strictly > 0).
        """

        # Check symmetry
        if not torch.allclose(A, A.T, atol=1e-6):
            raise ValueError(f"Matrix {name} is not symmetric.")

        # Eigenvalue check
        eigenvalues = torch.linalg.eigvalsh(A)
        min_eig = torch.min(eigenvalues)

        if min_eig <= 0:
            raise ValueError(
                f"Matrix {name} is not strictly positive definite. "
                f"Minimum eigenvalue = {min_eig.item():.6e}"
            )

    def pde_residual(self, t, x):
        t.requires_grad_(True)
        x.requires_grad_(True)

        u = self.net(t, x)

        # Time derivative
        u_t = torch.autograd.grad(
            u, t,
            grad_outputs=torch.ones_like(u),
            create_graph=True
        )[0]

        # First spatial gradient
        grad_u = torch.autograd.grad(
            u, x,
            grad_outputs=torch.ones_like(u),
            create_graph=True
        )[0]

        # Hessian (loop over dimensions)
        hessian = []
        for i in range(self.dim):
            grad_i = torch.autograd.grad(
                grad_u[:, i],
                x,
                grad_outputs=torch.ones_like(grad_u[:, i]),
                create_graph=True
            )[0]
            hessian.append(grad_i)

        hessian = torch.stack(hessian, dim=2)  # shape: (batch, dim, dim)

        # Diffusion term: 1/2 tr(σσ^T Hessian)
        sigma_sigmaT = self.sigma @ self.sigma.T
        diffusion = 0.5 * torch.einsum("ij,bji->b", sigma_sigmaT, hessian)

        # Drift gradient terms
        Hx = x @ self.H.T
        Malpha = self.alpha @ self.M.T

        drift_term = (grad_u * (Hx + Malpha)).sum(dim=1)

        # Quadratic source terms
        quad_x = torch.einsum("bi,ij,bj->b", x, self.C, x)
        quad_alpha = self.alpha @ self.D @ self.alpha

        residual = (
            u_t.squeeze()
            + diffusion
            + drift_term
            + quad_x
            + quad_alpha
        )

        return residual

    def loss(self, batch_size):

        # Sample interior points
        t = torch.rand(batch_size, 1, device=self.device) * self.T
        x = torch.randn(batch_size, self.dim, device=self.device)

        res = self.pde_residual(t, x)
        pde_loss = torch.mean(res**2)

        # Terminal condition
        x_T = torch.randn(batch_size, self.dim, device=self.device)
        t_T = torch.ones(batch_size, 1, device=self.device) * self.T

        u_T = self.net(t_T, x_T)
        target = torch.einsum("bi,ij,bj->b", x_T, self.R, x_T)

        terminal_loss = torch.mean((u_T.squeeze() - target)**2)

        return pde_loss + terminal_loss


    def train(self, epochs=5000, batch_size=256):
      """
      Train the model and record loss history and error against a given Monte Carlo mean.

      Parameters:
          epochs : number of training iterations
          batch_size : batch size for loss evaluation
      Returns:
          loss_history : list of training losses
      """

      self.loss_history = []

      for epoch in range(epochs):
          self.optimizer.zero_grad()

          # Compute loss on batch
          loss = self.loss(batch_size)
          loss.backward()

          self.optimizer.step()

          # Record loss
          self.loss_history.append(loss.item())

          if epoch % 250 == 249:
              print(f"Epoch {epoch}, Loss: {loss.item():.6f}")

      return self.loss_history

    def compute_hamiltonian_loss(theta_val_net, theta_act_net, H, M, C, D, batch_size=256, device="cpu"):
      """
      Compute the mean Hamiltonian for a batch of samples.

      Parameters
      ----------
      theta_val_net : nn.Module
          Value function network (fixed parameters).
      theta_act_net : nn.Module
          Policy network (parameters to optimise).
      H, M, C, D : torch.Tensor
          Problem matrices (dim x dim).
      batch_size : int
          Number of samples for estimating Hamiltonian.
      device : str
          'cpu' or 'cuda'.

      Returns
      -------
      torch.Tensor
          Mean Hamiltonian over the batch.
      """
      dim = H.shape[0]
      H = H.to(device)
      M = M.to(device)
      C = C.to(device)
      D = D.to(device)

      # Sample batch points
      t = torch.rand(batch_size, 1, device=device)
      x = torch.randn(batch_size, dim, device=device)
      x.requires_grad_(True)

      # Value function gradient
      v = theta_val_net(t, x)
      grad_v = torch.autograd.grad(
          v, x, grad_outputs=torch.ones_like(v), create_graph=True
      )[0]  # shape (batch, dim)

      # Current policy
      a = theta_act_net(t, x)  # shape (batch, dim)

      # Hamiltonian: H_i = grad_v^T H x + grad_v^T M a + x^T C x + a^T D a
      ham = (grad_v @ H.T) * x  # grad_v^T H x, shape (batch, dim)
      ham = ham.sum(dim=1)  # sum over dimensions
      ham += (grad_v @ M.T * a).sum(dim=1)  # grad_v^T M a
      ham += torch.einsum("bi,ij,bj->b", x, C, x)  # x^T C x
      ham += torch.einsum("bi,ij,bj->b", a, D, a)  # a^T D a

      return ham.mean()

    def policy_improvement(theta_val_net, theta_act_net, H, M, C, D, batch_size=256, device="cpu", lr=1e-3, n_steps=100):
      """
      Perform policy improvement: update theta_act to minimize Hamiltonian.

      Parameters
      ----------
      theta_val_net : nn.Module
          Value function network (fixed parameters).
      theta_act_net : nn.Module
          Policy network to update.
      H, M, C, D : torch.Tensor
          Problem matrices.
      batch_size : int
          Number of points to sample per step.
      device : str
          'cpu' or 'cuda'.
      lr : float
          Learning rate.
      n_steps : int
          Gradient steps to take.

      Returns
      -------
      theta_act_net : nn.Module
          Updated policy network.
      """
      optimizer = torch.optim.Adam(theta_act_net.parameters(), lr=lr)
      theta_val_net.eval()  # freeze value function

      for _ in range(n_steps):
          optimizer.zero_grad()
          loss = compute_hamiltonian_loss(theta_val_net, theta_act_net, H, M, C, D, batch_size, device)
          loss.backward()
          optimizer.step()

      return theta_act_net

if __name__ == "__main__":
